# ⚙️ The Optimizer: AdamW & Parameter Splitting Lab

Welcome! In this notebook, we master <b>AdamW</b>, <b>Decoupled Weight Decay</b>, and the critical rule of <b>Parameter Splitting</b> in foundation model pretraining.

---

## 1. The Core Metaphor: The Heavy Backpack vs. The Volume Knob

To understand why we cannot treat all parameters the same, consider the two kinds of weights in a Transformer:

<ol>
  <li><b>2D Matrix Projections (The Core Knowledge):</b><br>
  These are the big linear layers: <code>w_gate</code>, <code>w_up</code>, <code>w_down</code>, <code>q_proj</code>.<br>
  If left unchecked, their values explode to 100.0 or 1000.0 (overfitting to noise like a hiker with a heavy backpack).<br>
  &rarr; <b>Solution:</b> Apply <b>Weight Decay = 0.1</b> to gently pull them back toward zero at every step.</li>
  <br>
  <li><b>1D RMSNorm Scales (The Volume Knob):</b><br>
  RMSNorm has a learnable scale <b>gamma</b> initialized to 1.0. It sets the baseline volume of each layer.<br>
  If you apply Weight Decay to gamma, it will decay down toward <b>0.0</b>!<br>
  &rarr; <b>The Disaster:</b> Multiplying by 0.0 silences the entire layer, killing gradient flow completely.<br>
  &rarr; <b>Solution:</b> Weight Decay for gamma must ALWAYS be strictly <b>0.0 (OFF!)</b>.</li>
</ol>

---
## 2. Visual Architecture: The Parameter Split

<pre style='font-size: 13px; background: #0d1117; color: #58a6ff; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
                               All Model Parameters
                                        │
                    ┌───────────────────┴───────────────────┐
                    ▼                                       ▼
        Group 1: 2D Matrices                   Group 2: 1D Parameters
        (w_gate, w_up, q_proj, k_proj)         (RMSNorm scales, Biases)
                    │                                       │
                    ▼                                       ▼
       Weight Decay = 0.1 (ON)                 Weight Decay = 0.0 (OFF!)
       (Prevents weight explosion)             (Keeps volume knob alive)
</pre>

---
## 3. What is "Decoupled" Weight Decay in AdamW?

In classic Adam (pre-2017), weight decay was mixed into the gradient, which caused fast-moving weights to receive almost no decay.<br><br>
<b>AdamW (Loshchilov & Hutter) decoupled weight decay:</b><br>
At every step, after standard Adam momentum moves the weights, the weight is directly shrunk by:

<pre style='font-size: 15px; background: #161b22; color: #7ee787; padding: 12px; border-radius: 8px;'>
W_new = W_old - ( Learning_Rate · Weight_Decay · W_old )
</pre>

### Exercise 1: Pencil & Paper Calculation 📝
Suppose:
- Current weight: <b>W_old = 10.0</b>
- Learning rate: <b>η = 0.01</b>
- Weight decay: <b>λ = 0.1</b>

<b>Calculate on paper:</b>
1. Decay amount = 0.01 · 0.1 · 10.0 = <b>0.01</b>
2. W_new = 10.0 - 0.01 = <b>9.99</b><br><br>
<i>Notice: The weight shrunk by a controlled 0.1%, keeping it light and preventing explosion!</i>

---
## 4. Why beta2 = 0.95 Instead of 0.999 in Foundation Models?

Default PyTorch AdamW uses <code>betas = (0.9, 0.999)</code>.<br>
In foundation pretraining (Llama 3, DeepSeek, Mirendil-Base), researchers change beta2 to <b>0.95</b>!

<pre style='font-size: 14px; background: #0d1117; color: #c9d1d9; padding: 14px; border-radius: 8px; border: 1px solid #30363d;'>
Why beta2 = 0.95?
- beta2 tracks the moving average of gradient variance.
- 0.999 has a memory window of roughly 1 / (1 - 0.999) = 1,000 steps (too sluggish!).
- 0.95 has a memory window of roughly 1 / (1 - 0.95) = 20 steps.
- If a sudden spike in gradient variance occurs, beta2 = 0.95 adapts in 20 steps instead of taking 1,000 steps to recover!
</pre>

---
## 5. Verify Parameter Splitting with Real Project Optimizer Code

Now let's run our actual <code>configure_optimizers</code> function from <code>src/engine/optim.py</code> on our Transformer model and verify the parameter groups:

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.model.transformer import TransformerDecoder, ModelConfig
from src.engine.optim import configure_optimizers

# Instantiate micro model
config = ModelConfig(vocab_size=1000, dim=128, n_layers=2, n_heads=4, n_kv_heads=2)
model = TransformerDecoder(config)

# Configure optimizer with 0.1 weight decay
optimizer = configure_optimizers(model, weight_decay=0.1, learning_rate=1e-3, betas=(0.9, 0.95))

print(f"\nOptimizer Groups Created: {len(optimizer.param_groups)}")
for i, group in enumerate(optimizer.param_groups):
    sample_param = group['params'][0]
    print(f"Group {i+1}: Weight Decay = {group['weight_decay']:<4} | Parameter Count = {len(group['params']):<3} | Sample Shape = {list(sample_param.shape)}")

# Verify that Group 2 has strictly weight_decay == 0.0
assert optimizer.param_groups[1]['weight_decay'] == 0.0
print("\nSUCCESS: RMSNorm and 1D parameters are 100% protected with weight_decay = 0.0!")